## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 30 samples and 12123 columns.
First few rows of the data:


,OS.time,OS,Hypoxia_DEG_Score,Gender,Grouping,sample_title,disease state,tissue,treatment,SAMD11,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,2.504110,1,7.195420,Male,High,sample49,hepatocellular carcinoma,tumor,without treatment,2.130419,...,8.387673,7.346941,8.828142,8.576167,7.499479,7.363618,9.406952,7.648731,6.641186,8.360811
1,0.706849,1,46.831832,Male,High,sample67,hepatocellular carcinoma,tumor,without treatment,1.807826,...,8.131771,6.876079,8.426749,8.604203,7.020778,6.421280,8.775683,7.273592,6.598649,7.762321
2,5.087671,0,47.358300,Male,High,sample68,hepatocellular carcinoma,tumor,without treatment,2.453005,...,8.315868,6.125461,8.025927,8.263996,7.395243,5.763364,8.514467,6.373429,5.374012,7.467705
3,4.980822,0,-26.489002,Male,Low,sample69,hepatocellular carcinoma,tumor,without treatment,2.262247,...,9.497874,8.197639,9.706256,9.848717,8.269399,7.758245,10.048666,8.983378,8.456078,9.127606
4,4.975342,0,-45.116475,Male,Low,sample70,hepatocellular carcinoma,tumor,without treatment,2.169161,...,9.350911,8.134647,9.595574,9.734664,7.856962,7.492528,9.689079,8.695094,7.941265,9.449073


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
